# Notebook de pruebas

En este notebook, se realizarán pruebas y experimentaciones del backend del proyecto.

1. Transcripción - Whisper.cpp
* Se obtendrán las transcripciones de varios audios.

2. Traducción a glosas
* A partir de las transcripciones, se realizará la traducción a glosas usando ruLSE.

3. Procesamiento de glosas
* Partiendo de las secuencias de glosas generadas, se observarán la secuencia de IDs de animaciones devuelta.

4. Captura de movimiento - MediaPipe
* Se comprobarán cómo afecta el ajuste de parámetros.
* Se probarán los filtros de suavizado de One Euro y Kalman con distintas combinaciones de parámetros.

## 1. Transcripción

In [ ]:
import requests

def transcribir_audio(ruta_audio):
    url = "http://127.0.0.1:8080/inference"
    
    with open(ruta_audio, "rb") as f:
        files = {"file": f}
        data = {
            "language": "es",
        }
        response = requests.post(url, files=files, data=data)
        return response.json()["text"]
    

audio0 = "./audios/audio0.wav" # "Hola, ¿cómo estás? Soy ramona, un avatar de lengua de signos. Puedo decir cualquier frase."
audio1 = "./audios/audio1.wav" # "¡Vaya baya hay encima de la valla!"
audio2 = "./audios/audio2.wav" # "El niño ayer salió a correr por el caminito del campo. 
                                        # Sin embargo, no esperaba encontrarse con Juan, su amigo de la infancia. 
                                        # Se pusieron a charlar y de la nada, apareció una liebre que les invitó a seguirla. 
                                        # Aunque sorprendidos por el acontecimiento, decidieron seguirla." 
audio3 = "./audios/audio3.wav" # "La chica se sentó en la silla porque estaba cansada"
audio4 = "./audios/audio4.wav" # "Tres tristes tigres comían trigo en un trigal"
audio5 = "./audios/audio5.wav" # "Pensé que lo había hecho todo, pero me faltó echar a la vaca en la baca del coche así que tuvo que venir Andrea, mi amiga fontanera, corriendo hasta aquí para ayudarme. 
                                        # Luego me reí con ella, porque con las prisas no tuvo tiempo de soltar el tubo del trabajo"

audios = [audio0, audio1, audio2, audio3, audio4, audio5]

for audio in range(len(audios)):
    transcripcion = transcribir_audio(audios[audio])
    print(f"- Transcripción del audio número {audio}: {transcripcion}\n")


- Transcripción del audio número 0:  Hola, ¿cómo estás? Soy Ramona, un avatar de lengua de signos.
 Puedo decir cualquier frase.


- Transcripción del audio número 1:  Vaya, vaya hay encima de la valla.


- Transcripción del audio número 2:  El niño ayer salió a correr por el caminito del campo.
 Sin embargo, no esperaba encontrarse con Juan, su amigo de la infancia.
 Se pusieron a charlar y de la nada apareció una liebre que les invitó a seguirla.
 Aunque sorprendidos por el acontecimiento, decidieron seguirla.


- Transcripción del audio número 3:  La chica se sentó en la silla porque estaba cansada.


- Transcripción del audio número 4:  Tres tristes tigres comían trigo en un trigal.


- Transcripción del audio número 5:  Pensé que lo había hecho todo, pero me faltó echar a la vaca en la vaca del coche,
 así que tuvo que venir Andrea, mi amiga fontanera, corriendo hasta aquí para ayudarme.
 Luego me reí con ella, porque con las prisas no tuvo tiempo de soltar el tubo del trabajo.




## 2. Traducción a glosas

In [1]:
import sys
sys.path.append('..')
from modelo.lse_simple import generatorText2Gloss
import stanza

nlp = stanza.Pipeline(lang='es', processors='tokenize,mwt,pos,lemma')

/Users/i/miniconda3/envs/avatar_lse_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-20 19:49:08 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-20 19:49:08 INFO: Downloaded file to /Users/i/stanza_resources/resources.json
2026-06-20 19:49:08 INFO: Loading these models for language: es (Spanish):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-06-20 19:49:08 INFO: Using device: cpu
2026-06-20 19:49:08 INFO: Loading: tokenize
2026-06-20 19:49:09 INFO: Loading: mwt
2026-06-20 19:49:09

In [ ]:
frase0 = "Hola, ¿cómo estás? Soy ramona, un avatar de lengua de signos. Puedo decir cualquier frasecita."
frase1 = "¿Has visto ya el audio que te envié?"
frase2 = "¡Gracias por verme y adiós!"
frase3 = "La chica se sentará en la silla porque estaba cansada"
frase4 = "Él vino a por vino cuando tú viniste"
frase5 = "Pensé que lo había hecho todo, pero me faltó echar a la vaca en la baca del coche así que tuvo que venir Andrea, mi amiga fontanera, corriendo hasta aquí para ayudarme. Luego me reí con ella, porque con las prisas no tuvo tiempo de soltar el tubo del trabajo"

frases = [frase0, frase1, frase2, frase3, frase4, frase5]

glosas = []

for frase in range(len(frases)):
    resultado_lse = generatorText2Gloss(frases[frase], "../modelo/rules.csv", "./corpus-generado.csv", nlp)
    glosas.append(resultado_lse)
    print(f"Texto original de la frase {frase}: {frases[frase]}")
    print(f"Glosa LSE de la frase {frase}: {resultado_lse}\n")

Texto original de la frase 0: Hola, ¿cómo estás? Soy ramona, un avatar de lengua de signos. Puedo decir cualquier frasecita.
Glosa LSE de la frase 0: TÚ HOLA CÓMO  YO RAMONA AVATAR LENGUA SIGNO-PL  YO FRASECITA CUALQUIERA DECIR PODER

Texto original de la frase 1: ¿Has visto ya el audio que te envié?
Glosa LSE de la frase 1: YO HABER YO AUDIO QUE TÚ ENVIAR VER YA

Texto original de la frase 2: ¡Gracias por verme y adiós!
Glosa LSE de la frase 2: GRACIA ADIÓS VER YO

Texto original de la frase 3: La chica se sentará en la silla porque estaba cansada
Glosa LSE de la frase 3: FUTURO CHICA ÉL SENTAR SILLA PORQUE CANSADO

Texto original de la frase 4: Él vino a por vino cuando tú viniste
Glosa LSE de la frase 4: PASADO YO ÉL VENIR VINO CUANDO TÚ VINISTIR

Texto original de la frase 5: Pensé que lo había hecho todo, pero me faltó echar a la vaca en la baca del coche así que tuvo que venir Andrea, mi amiga fontanera, corriendo hasta aquí para ayudarme. Luego me reí con ella, porque con las pr

## 3. Procesamiento de glosas

In [4]:
import os

# crear un diccionario con formato {palabra:id_animación} a partir de palabras_definidas.txt
diccionario = {}
diccionario_palabras = "../modelo/palabras_definidas.txt"

def cargar_diccionario():
    palabras = []
    if not os.path.exists(diccionario_palabras):
        print(f"ERROR: No existe {diccionario_palabras}")
        return

    with open(diccionario_palabras, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith("#"): continue
            
            # formato: palabra=animacion
            partes = linea.split("=")
            if len(partes) == 2:
                palabra = partes[0].strip().lower()
                anim_id = partes[1].strip()
                diccionario[palabra] = anim_id
                palabras.append(palabra)
    print(f"Diccionario creado. {len(diccionario)} entradas: {palabras}")

cargar_diccionario()

# obtener una secuencia final con todas los ids de las animaciones ordenados a partir de las glosas
def procesar_frase(frase):
    print(f"Frase LSE: {frase}")
    
    # Sustituir caracteres acentuados por sus equivalentes sin acento
    acentos = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U'
    }
    frase_normalizada = ''.join(acentos.get(c, c) for c in frase)
    print(f"Frase normalizada: {frase_normalizada}")
    
    secuencia_final = []
    palabras = frase_normalizada.split(" ")
    
    for palabra in palabras:
        palabra = palabra.split("-")[0] # en la notación, a veces se añade un "-", quitarlo
        palabra_limpia = palabra.strip().lower()
        if not palabra_limpia:  # frase vacía
            continue
        
        # si la palabra está definida, se procesa completa
        if palabra_limpia in diccionario:
            secuencia_final.append(diccionario[palabra_limpia])
        # si la palabra no está definida, se deletrea
        else:
            for letra in palabra_limpia:
                if letra in diccionario:
                    secuencia_final.append(diccionario[letra])
                else:
                    print(f"No existe animación para la letra '{letra}'")
    
    mensaje_final = ",".join(secuencia_final)
    return mensaje_final

Diccionario creado. 43 entradas: ['a', 'adiós', 'b', 'c', 'cómo', 'cualquiera', 'd', 'decir', 'e', 'este', 'f', 'frase', 'g', 'gracia', 'h', 'hola', 'i', 'idle', 'j', 'k', 'l', 'lengua', 'm', 'n', 'o', 'p', 'poder', 'q', 'r', 's', 'signo', 't', 'tú', 'u', 'v', 'ver', 'vídeo', 'video', 'w', 'x', 'y', 'yo', 'z']


In [8]:
for frase in range(len(glosas)):
    glosa_procesada = procesar_frase(glosas[frase])
    print(f"Glosa LSE procesada de la frase {frase}: {glosa_procesada}\n")

Frase LSE: TÚ HOLA CÓMO  YO RAMONA AVATAR LENGUA SIGNO-PL  YO FRASECITA CUALQUIERA DECIR PODER
Frase normalizada: TU HOLA COMO  YO RAMONA AVATAR LENGUA SIGNO-PL  YO FRASECITA CUALQUIERA DECIR PODER
Glosa LSE procesada de la frase 0: t_anim,u_anim,hola_anim,c_anim,o_anim,m_anim,o_anim,yo_anim,r_anim,a_anim,m_anim,o_anim,n_anim,a_anim,a_anim,v_anim,a_anim,t_anim,a_anim,r_anim,lengua_anim,signo_anim,yo_anim,f_anim,r_anim,a_anim,s_anim,e_anim,c_anim,i_anim,t_anim,a_anim,cualquiera_anim,decir_anim,poder_anim

Frase LSE: YO HABER YO AUDIO QUE TÚ ENVIAR VER YA
Frase normalizada: YO HABER YO AUDIO QUE TU ENVIAR VER YA
Glosa LSE procesada de la frase 1: yo_anim,h_anim,a_anim,b_anim,e_anim,r_anim,yo_anim,a_anim,u_anim,d_anim,i_anim,o_anim,q_anim,u_anim,e_anim,t_anim,u_anim,e_anim,n_anim,v_anim,i_anim,a_anim,r_anim,ver_anim,y_anim,a_anim

Frase LSE: GRACIA ADIÓS VER YO
Frase normalizada: GRACIA ADIOS VER YO
Glosa LSE procesada de la frase 2: gracias_anim,a_anim,d_anim,i_anim,o_anim,s_anim,ver_ani

## 4. MediaPipe

### Función de dibujo

In [9]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import json
import os
import numpy as np

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)

conexiones_poses = [
    (11, 12), (11, 13), (12, 14), (13,15), (14,16), # Brazos hasta codo
    (11, 23), (12, 24), (23, 24), # Tronco
    (23, 25), (24, 26), (25, 27), (26, 28), # Piernas
    (27, 29), (29, 31), (28, 30), (30, 32)  # Pies
]

puntos_excluidos = list(range(0,11)) + list(range(17,23))

def draw_complete_skeleton(rgb_image, resultado_pose, resultado_mano):
    annotated_image = np.copy(rgb_image)
    h, w, _ = annotated_image.shape

    if resultado_mano and resultado_mano.hand_landmarks:
        hand_landmarks_list = resultado_mano.hand_landmarks
        handedness_list = resultado_mano.handedness
        for idx in range(len(hand_landmarks_list)):
            hand_landmarks = hand_landmarks_list[idx]
            handedness = handedness_list[idx]

            mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

            height, width, _ = annotated_image.shape
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * width)
            text_y = int(min(y_coordinates) * height) - MARGIN

            cv2.putText(annotated_image, f"{handedness[0].category_name}",
                        (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                        FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

    if resultado_pose and resultado_pose.pose_landmarks:
        conexiones_poses_copia = list(conexiones_poses)
        puntos_excluidos_copia = list(puntos_excluidos)
        for landmarks in resultado_pose.pose_landmarks:
            for idx, lm in enumerate(landmarks):
                if idx in puntos_excluidos_copia: 
                    continue
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(annotated_image, (cx, cy), 3, (0, 255, 0), -1)

            for a, b in conexiones_poses_copia:
                p1, p2 = landmarks[a], landmarks[b]
                cv2.line(annotated_image, 
                         (int(p1.x * w), int(p1.y * h)), 
                         (int(p2.x * w), int(p2.y * h)), 
                         (255, 0, 0), 2)


    return annotated_image

### Modelo sin ningún filtro de suavizado

In [ ]:
# opciones de configuración ------------------------------------------------

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path_poselandmarker = "../landmarkers/pose_landmarker_full.task"
model_path_handlandmarker = "../landmarkers/hand_landmarker.task"


options_pose_1 = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.4,
    min_pose_presence_confidence=0.4,
    min_tracking_confidence=0.4
)

options_pose_2 = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.6,
    min_pose_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

options_pose_3 = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.8,
    min_pose_presence_confidence=0.8,
    min_tracking_confidence=0.8
)

options_hand_1 = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.4,
    min_hand_presence_confidence=0.4,
    min_tracking_confidence=0.4
)

options_hand_2 = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

options_hand_3 = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.8,
    min_hand_presence_confidence=0.8,
    min_tracking_confidence=0.8
)

options_hand = [options_hand_1, options_hand_2, options_hand_3]
options_pose = [options_pose_1, options_pose_2, options_pose_3]

# ------------------------------------------------------------------------

indices_a_incluir = list(range(0,33))

videos_folder = "./videos_prueba"
extensiones = {'avi', 'mov', 'mp4'}

for option_idx in range(len(options_pose)):

    if not os.path.exists(f"./videos_marcados_clean/{option_idx}"): os.makedirs(f"./videos_marcados_clean/{option_idx}")
    if not os.path.exists(f"./animaciones_json_clean/{option_idx}"): os.makedirs(f"./animaciones_json_clean/{option_idx}")

    with os.scandir(videos_folder) as videos:
        for video in videos:
            video_full_name = os.path.split(video)[1].split('.')
            if video.is_file() and video_full_name[1] in extensiones:
                video_name = video_full_name[0]
                output_video = f"./videos_marcados_clean/{option_idx}/{video_name}_clean.mp4"

                cap = cv2.VideoCapture(video.path)
                fps = cap.get(cv2.CAP_PROP_FPS)
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

                fps = fps if fps > 0 else 30.0

                writer = cv2.VideoWriter(
                    output_video,
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    fps,
                    (w, h)
                )

                animacion_esqueleto = []

                # marcadores
                with PoseLandmarker.create_from_options(options_pose[option_idx]) as pose_detect, \
                    HandLandmarker.create_from_options(options_hand[option_idx]) as hand_detect:

                    frame_idx = 0
                    while cap.isOpened():
                        ret, frame = cap.read()
                        if not ret:
                            break

                        frame_timestamp_ms = int((frame_idx / fps) * 1000)
                        t_seconds = frame_timestamp_ms / 1000.0
                        
                        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                        resultado_pose = pose_detect.detect_for_video(mp_image, frame_timestamp_ms)
                        resultado_mano = hand_detect.detect_for_video(mp_image, frame_timestamp_ms)

                        datos_frame = {
                            "frame": frame_idx,
                            "timestamp": frame_timestamp_ms,
                            "pose": [],
                            "hands": []
                        }

                        # manos
                        if resultado_mano.hand_landmarks:
                            for i, hand in enumerate(resultado_mano.hand_landmarks):
                                lado = resultado_mano.handedness[i][0].category_name
                                puntos_list = []
                                for idx_pt, l in enumerate(hand):
                                    puntos = {'x': l.x, 'y': l.y, 'z': l.z}
                                    puntos_list.append(puntos)
                                datos_frame["hands"].append({"lado": lado, "puntos": puntos_list})

                        # pose
                        if resultado_pose.pose_landmarks:
                            p = resultado_pose.pose_landmarks[0]
                            for idx in indices_a_incluir:
                                puntos = {'x': p[idx].x, 'y': p[idx].y, 'z': p[idx].z}
                                datos_frame["pose"].append(puntos)

                        animacion_esqueleto.append(datos_frame)

                        # dibujar resultados
                        annotated_frame = draw_complete_skeleton(frame_rgb, resultado_pose, resultado_mano)
                        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
                        writer.write(final_frame)

                        frame_idx += 1

                cap.release()
                writer.release()

                json_folder = f"./animaciones_json_clean/{option_idx}/{video_name}_clean.json"
                with open(json_folder, "w") as f:
                    json.dump(animacion_esqueleto, f, indent=4)

                print(f"Animación sin filtrado con configuración {option_idx} de {video_name} guardada.")


I0000 00:00:1781452544.734461  212836 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452544.793016  212839 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452544.806028  212842 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452544.818433  212856 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452544.822200  212858 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452544.829139  212858 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación sin filtrado con configuración 0 de video guardada.


I0000 00:00:1781452545.905255  212955 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452545.955843  212957 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452545.962151  212957 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452545.965602  212978 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452545.967969  212984 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452545.971459  212983 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación sin filtrado con configuración 0 de cualquiera guardada.


I0000 00:00:1781452546.875925  213065 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452546.927183  213068 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452546.933962  213071 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452546.937286  213084 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452546.939393  213087 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452546.941898  213089 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

Animación sin filtrado con configuración 2 de cualquiera guardada.


I0000 00:00:1781452554.387898  213896 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452554.435555  213901 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452554.442159  213903 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452554.454088  213916 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452554.456480  213919 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452554.459231  213919 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación sin filtrado con configuración 2 de z guardada.


I0000 00:00:1781452555.217546  213988 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452555.267573  213990 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452555.273886  214002 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452555.277175  214005 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452555.279296  214007 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452555.281883  214011 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación sin filtrado con configuración 2 de como guardada.


### Modelo con filtro OneEuro

In [11]:
import math

def smoothing_factor(t_e, cutoff):
    """
    Calcula el factor de suavizado (alpha) basándose en la frecuencia de corte y el tiempo transcurrido (t_e)
    """
    r = 2 * math.pi * cutoff * t_e
    return r / (r + 1)


def exponential_smoothing(a, x, x_prev):
    """
    Aplica la fórmula estándar de un filtro de paso bajo (suavizado exponencial)
    """
    return a * x + (1 - a) * x_prev


class OneEuroFilter:
    def __init__(self, t0, x0, dx0=0.0, min_cutoff=1.0, beta=0.0,d_cutoff=1.0):
        self.min_cutoff = float(min_cutoff)
        self.beta = float(beta)
        self.d_cutoff = float(d_cutoff)
        self.x_prev = float(x0)
        self.dx_prev = float(dx0)
        self.t_prev = float(t0)

    def __call__(self, t, x):
        t_e = t - self.t_prev   #tiempo transcurrido desde el último dato

        if t_e <= 0:    # si el tiempo retrocede o no avanza, se devuelve la posición guardada
            return self.x_prev

        a_d = smoothing_factor(t_e, self.d_cutoff)
        dx = (x - self.x_prev) / t_e    # velocidad cruda
        dx_hat = exponential_smoothing(a_d, dx, self.dx_prev)   # velocidad suavizada

        cutoff = self.min_cutoff + self.beta * abs(dx_hat)
        a = smoothing_factor(t_e, cutoff)
        x_hat = exponential_smoothing(a, x, self.x_prev)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t

        return x_hat

In [ ]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path_poselandmarker = "../landmarkers/pose_landmarker_full.task"
model_path_handlandmarker = "../landmarkers/hand_landmarker.task"

options_pose = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.6,
    min_pose_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

options_hand = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

indices_a_incluir = list(range(0,33))

# parámetros 1 euro filter
CFG_1EURO_1 = {'min_cutoff': 0.01, 'beta': 1, 'd_cutoff': 1.0}  # movimiento detectado bajo y se prioriza suavidad
CFG_1EURO_2 = {'min_cutoff': 0.1, 'beta': 5, 'd_cutoff': 1.0}   # movimiento detectado promedio y prioridad suavidad-velocidad compensada
CFG_1EURO_3 = {'min_cutoff': 0.5, 'beta': 10, 'd_cutoff': 1.0}  # movimiento detectado alto y se prioriza velocidad

CFG_1EURO = [CFG_1EURO_1, CFG_1EURO_2, CFG_1EURO_3]

videos_folder = "./videos_prueba"
extensiones = {'avi', 'mov', 'mp4'}

for cfg_idx in range(len(CFG_1EURO)):

    if not os.path.exists(f"./videos_marcados_oneeuro/{cfg_idx}"): os.makedirs(f"./videos_marcados_oneeuro/{cfg_idx}")
    if not os.path.exists(f"./animaciones_json_oneeuro/{cfg_idx}"): os.makedirs(f"./animaciones_json_oneeuro/{cfg_idx}")

    with os.scandir(videos_folder) as videos:
        for video in videos:
            video_full_name = os.path.split(video)[1].split('.')
            if video.is_file() and video_full_name[1] in extensiones:
                video_name = video_full_name[0]
                output_video = f"./videos_marcados_oneeuro/{cfg_idx}/{video_name}_oneeuro.mp4"

                cap = cv2.VideoCapture(video.path)
                fps = cap.get(cv2.CAP_PROP_FPS)
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

                fps = fps if fps > 0 else 30.0

                writer = cv2.VideoWriter(
                    output_video,
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    fps,
                    (w, h)
                )

                animacion_esqueleto = []
                filtros_euro = {} # estado de los filtros

                # marcadores
                with PoseLandmarker.create_from_options(options_pose) as pose_detect, \
                    HandLandmarker.create_from_options(options_hand) as hand_detect:

                    frame_idx = 0
                    while cap.isOpened():
                        ret, frame = cap.read()
                        if not ret:
                            break

                        frame_timestamp_ms = int((frame_idx / fps) * 1000)
                        t_seconds = frame_timestamp_ms / 1000.0
                        
                        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                        resultado_pose = pose_detect.detect_for_video(mp_image, frame_timestamp_ms)
                        resultado_mano = hand_detect.detect_for_video(mp_image, frame_timestamp_ms)

                        datos_frame = {
                            "frame": frame_idx,
                            "timestamp": frame_timestamp_ms,
                            "pose": [],
                            "hands": []
                        }

                        # manos
                        if resultado_mano.hand_landmarks:
                            for i, hand in enumerate(resultado_mano.hand_landmarks):
                                lado = resultado_mano.handedness[i][0].category_name
                                puntos_unreal = []
                                for idx_pt, l in enumerate(hand):
                                    ejes = {'x': l.x, 'y': l.y, 'z': l.z}
                                    res_filtrado = {}

                                    for axis, val in ejes.items():
                                        key = f"hand_{lado}_{idx_pt}_{axis}"
                                        if key not in filtros_euro:
                                            filtros_euro[key] = OneEuroFilter(t_seconds, val, **CFG_1EURO[cfg_idx])
                                            res_filtrado[axis] = val
                                        else:
                                            res_filtrado[axis] = filtros_euro[key](t_seconds, val)

                                    puntos_unreal.append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})
                                    
                                    # actualizar valores del marcador con los valores suavizados
                                    l.x = res_filtrado['x']
                                    l.y = res_filtrado['y']
                                    l.z = res_filtrado['z']

                                datos_frame["hands"].append({"lado": lado, "puntos": puntos_unreal})

                        # pose
                        if resultado_pose.pose_landmarks:
                            p = resultado_pose.pose_landmarks[0]
                            for idx in indices_a_incluir:
                                ejes = {'x': p[idx].x, 'y': p[idx].y, 'z': p[idx].z}
                                res_filtrado = {}

                                for axis, val in ejes.items():
                                    key = f"pose_{idx}_{axis}"
                                    if key not in filtros_euro:
                                        filtros_euro[key] = OneEuroFilter(t_seconds, val, **CFG_1EURO[cfg_idx])
                                        res_filtrado[axis] = val
                                    else:
                                        res_filtrado[axis] = filtros_euro[key](t_seconds, val)
                                
                                datos_frame["pose"].append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})

                                p[idx].x = res_filtrado['x']
                                p[idx].y = res_filtrado['y']
                                p[idx].z = res_filtrado['z']

                        animacion_esqueleto.append(datos_frame)

                        # dibujar resultados
                        annotated_frame = draw_complete_skeleton(frame_rgb, resultado_pose, resultado_mano)
                        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
                        writer.write(final_frame)

                        frame_idx += 1

                cap.release()
                writer.release()

                # exportar resultados en JSON
                json_folder = f"./animaciones_json_oneeuro/{cfg_idx}/{video_name}_oneeuro.json"
                with open(json_folder, "w") as f:
                    json.dump(animacion_esqueleto, f, indent=4)

                print(f"Animación 1Euro con configuración {cfg_idx} de {video_name} guardada.")


I0000 00:00:1781452556.122566  214079 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452556.176309  214080 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452556.182356  214089 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452556.185501  214098 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452556.187807  214100 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452556.191572  214102 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro con configuración 0 de video guardada.


I0000 00:00:1781452557.306938  214194 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452557.360215  214196 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452557.365996  214204 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452557.368765  214217 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452557.371217  214220 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452557.373790  214220 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro con configuración 0 de cualquiera guardada.


I0000 00:00:1781452558.301833  214288 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452558.351000  214291 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452558.357051  214293 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452558.360565  214305 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452558.362894  214307 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452558.365390  214307 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

Animación 1Euro con configuración 2 de cualquiera guardada.


I0000 00:00:1781452565.960642  215006 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452566.010890  215010 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452566.017005  215010 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452566.020300  215023 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452566.022732  215025 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452566.025054  215027 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro con configuración 2 de z guardada.


I0000 00:00:1781452566.719249  215098 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452566.770444  215099 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452566.777055  215101 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452566.785860  215116 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452566.788180  215118 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452566.791248  215118 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro con configuración 2 de como guardada.


### Modelo con filtro Kalman

In [ ]:
class KalmanFilter:
    def __init__(self, t0, x0, Q_val=1e-5, R_val=1e-2):
        self.t_prev = float(t0)
        
        # Dimensiones del sistema
        self.n = 2 # [Posición, Velocidad]
        self.m = 1 # [Posición]
        
        # Estado inicial (x)
        self.x = np.array([[float(x0)], # posición detectada
                           [0.0]])      # velocidad 0
        
        self.P = np.eye(self.n)             # matriz de covarianza del error (P)
        self.H = np.array([[1.0, 0.0]])     # matriz de observación (H)
        self.Q = np.array([[Q_val, 0.0], 
                           [0.0, Q_val]])   # ruido del proceso (Q)
        self.R = np.array([[R_val]])        # ruido de la medición (R)
        self.I = np.eye(self.n)             # matriz Identidad para los cálculos

    def __call__(self, t, z_val):
        dt = t - self.t_prev
        if dt <= 0:
            return float(self.x[0, 0])

        F = np.array([[1.0, dt], 
                      [0.0, 1.0]])  # matriz de transición de estado (F)
        
        self.x = np.dot(F, self.x)
        self.P = np.dot(np.dot(F, self.P), F.T) + self.Q

        z = np.array([[float(z_val)]])      # se convierten los datos de MediaPipe a matriz
        y = z - np.dot(self.H, self.x)      # error entre medición y predicción
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R   # incertidumbre total (S)
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))  # ganancia de Kalman (K)
        
        self.x = self.x + np.dot(K, y)
        self.P = np.dot((self.I - np.dot(K, self.H)), self.P)
        self.t_prev = t
        
        return float(self.x[0, 0])

In [ ]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path_poselandmarker = "../landmarkers/pose_landmarker_full.task"
model_path_handlandmarker = "../landmarkers/hand_landmarker.task"

options_pose = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.6,
    min_pose_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

options_hand = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

indices_a_incluir = list(range(0,33))

CFG_KALMAN_1 = {'Q_val': 1e-6, 'R_val': 10.0}  # muy suave, pero los movimientos rápidos tendrán retraso
CFG_KALMAN_2 = {'Q_val': 1e-4, 'R_val': 1.0}   # modo equilibrado
CFG_KALMAN_3 = {'Q_val': 1e-2, 'R_val': 0.01}  # movimiento instantáneo pero tembloroso

CFG_KALMAN = [CFG_KALMAN_1, CFG_KALMAN_2, CFG_KALMAN_3]

videos_folder = "./videos_prueba"
extensiones = {'avi', 'mov', 'mp4'}

for cfg_idx in range(len(CFG_KALMAN)):

    if not os.path.exists(f"./videos_marcados_kalman/{cfg_idx}"): os.makedirs(f"./videos_marcados_kalman/{cfg_idx}")
    if not os.path.exists(f"./animaciones_json_kalman/{cfg_idx}"): os.makedirs(f"./animaciones_json_kalman/{cfg_idx}")

    with os.scandir(videos_folder) as videos:
        for video in videos:
            video_full_name = os.path.split(video)[1].split('.')
            if video.is_file() and video_full_name[1] in extensiones:
                video_name = video_full_name[0]
                output_video = f"./videos_marcados_kalman/{cfg_idx}/{video_name}_kalman.mp4"

                cap = cv2.VideoCapture(video.path)
                fps = cap.get(cv2.CAP_PROP_FPS)
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                fps = fps if fps > 0 else 30.0

                writer = cv2.VideoWriter(
                    output_video,
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    fps,
                    (w, h)
                )

                animacion_esqueleto = []
                filtros_kalman = {}

                with PoseLandmarker.create_from_options(options_pose) as pose_detect, \
                    HandLandmarker.create_from_options(options_hand) as hand_detect:

                    frame_idx = 0
                    while cap.isOpened():
                        ret, frame = cap.read()
                        if not ret:
                            break

                        frame_timestamp_ms = int((frame_idx / fps) * 1000)
                        t_seconds = frame_timestamp_ms / 1000.0
                        
                        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                        resultado_pose = pose_detect.detect_for_video(mp_image, frame_timestamp_ms)
                        resultado_mano = hand_detect.detect_for_video(mp_image, frame_timestamp_ms)

                        datos_frame = {"frame": frame_idx, "timestamp": frame_timestamp_ms, "pose": [], "hands": []}

                        # manos
                        if resultado_mano.hand_landmarks:
                            for i, hand in enumerate(resultado_mano.hand_landmarks):
                                lado = resultado_mano.handedness[i][0].category_name
                                puntos_unreal = []
                                for idx_pt, l in enumerate(hand):
                                    ejes = {'x': l.x, 'y': l.y, 'z': l.z}
                                    res_filtrado = {}

                                    for axis, val in ejes.items():
                                        key = f"hand_{lado}_{idx_pt}_{axis}"
                                        if key not in filtros_kalman:
                                            filtros_kalman[key] = KalmanFilter(t_seconds, val, **CFG_KALMAN[cfg_idx])
                                            res_filtrado[axis] = val
                                        else:
                                            res_filtrado[axis] = filtros_kalman[key](t_seconds, val)

                                    puntos_unreal.append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})
                                    l.x, l.y, l.z = res_filtrado['x'], res_filtrado['y'], res_filtrado['z']

                                datos_frame["hands"].append({"lado": lado, "puntos": puntos_unreal})

                        # pose
                        if resultado_pose.pose_landmarks:
                            p = resultado_pose.pose_landmarks[0]
                            for idx_pt in indices_a_incluir:
                                ejes = {'x': p[idx_pt].x, 'y': p[idx_pt].y, 'z': p[idx_pt].z}
                                res_filtrado = {}

                                for axis, val in ejes.items():
                                    key = f"pose_{idx_pt}_{axis}"
                                    if key not in filtros_kalman:
                                        filtros_kalman[key] = KalmanFilter(t_seconds, val, **CFG_KALMAN[cfg_idx])
                                        res_filtrado[axis] = val
                                    else:
                                        res_filtrado[axis] = filtros_kalman[key](t_seconds, val)
                                
                                datos_frame["pose"].append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})
                                p[idx_pt].x, p[idx_pt].y, p[idx_pt].z = res_filtrado['x'], res_filtrado['y'], res_filtrado['z']

                        animacion_esqueleto.append(datos_frame)

                        annotated_frame = draw_complete_skeleton(frame_rgb, resultado_pose, resultado_mano)
                        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
                        writer.write(final_frame)

                        frame_idx += 1

                cap.release()
                writer.release()

                json_folder = f"./animaciones_json_kalman/{cfg_idx}/{video_name}_kalman.json"
                with open(json_folder, "w") as f:
                    json.dump(animacion_esqueleto, f, indent=4)

                print(f"Animación Kalman con configuración {cfg_idx} de {video_name} guardada.")

I0000 00:00:1781452567.638104  215184 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452567.688982  215187 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452567.694872  215187 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452567.702773  215203 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452567.705147  215205 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452567.707619  215205 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación Kalman con configuración 0 de video guardada.


I0000 00:00:1781452568.837522  215304 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452568.887000  215306 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452568.893214  215309 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452568.902354  215322 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452568.904544  215324 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452568.907329  215324 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación Kalman con configuración 0 de cualquiera guardada.


I0000 00:00:1781452569.836550  215398 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452569.889745  215400 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452569.895319  215400 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452569.900283  215415 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452569.902278  215417 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452569.904692  215420 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

Animación Kalman con configuración 2 de cualquiera guardada.


I0000 00:00:1781452577.520659  216215 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452577.569635  216216 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452577.576168  216228 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452577.586409  216232 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452577.588670  216235 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452577.591130  216235 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación Kalman con configuración 2 de z guardada.


I0000 00:00:1781452578.285846  216291 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452578.335751  216294 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452578.341181  216297 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1781452578.344588  216308 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1781452578.346693  216310 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781452578.349301  216315 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación Kalman con configuración 2 de como guardada.
